# R13 benign — lens-conditioned IFEval paired with the harmful R12 grid

unsloth/Meta-Llama-3.1-8B-Instruct · the existing harmful 7-lens grid ·
the fixed first 50 IFEval prompts under exactly five benign system-prompt lenses.

This outcome stays outside analyze_dual_primary.py. Each trained cell evaluates the
harmful grid and the benign grid from the same retained round-3 model path, copies the
harmful result fingerprint into the benign manifest with that provenance, then deletes
the temporary checkpoint. The benign runner does not independently recompute a fingerprint.

Unique output suffix: _r13_benign_env76. Existing R12 and old IFEval outputs are never reused.

This file is generated by src/make_nb_r13_benign.py; do not edit it directly.

## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
비교 가능성 핀: `unsloth==2026.7.6`, `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth==2026.7.6")
pip("transformers==4.56.2", "trl==0.22.2")

from importlib.metadata import version
_installed_unsloth = version("unsloth")
assert _installed_unsloth == "2026.7.6", (
    f"Unsloth 설치 핀 불일치: 기대=2026.7.6 실제={_installed_unsloth}")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
pip("langdetect==1.0.9")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True,
                probe_rounds=None, max_new_tokens=256, out_suffix="", save_adapters=""):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다.
       probe_rounds — ★R4 신설. 프로브가 `r in eval_rounds`에 묶여 있어서
                      **eval 없이 프로브만** 뜰 수 없었다. 기하의 라운드 궤적을 재려면
                      eval(렌즈당 25분)을 다시 돌릴 이유가 없다 — 시드42 eval은 이미 있다.
                      None이면 eval_rounds를 따른다(기존 동작 그대로).
       max_new_tokens, out_suffix — ★R5 신설. 512-token 평가는 기존 256-token
                      결과와 파일명이 절대 겹치지 않게 접미사를 함께 쓴다."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    pr_rounds = eval_rounds if probe_rounds is None else probe_rounds
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}"
            + (f" --save_adapter_dir {save_adapters}/{arm}_r{r}_s{seed}" if save_adapters else ""))

        # ★R5: 라운드 r 머지본이 생긴 순간 r-1은 죽은 무게다. 원래는 eval/probe/mmlu가
        #   끝난 뒤에야 지웠는데, 4렌즈 eval은 약 2시간이라 그동안 32GB를 붙들고 있었다.
        #   R5 배치 2가 실제로 여기서 디스크를 터뜨렸다(Docker daemon 오류는 그 증상).
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제(학습 직후): {prev_dir}")
            prev_dir = None

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}{eval_args}")
                check(f"results/eval_{arm}_r{f}_s{seed}{out_suffix}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
            check(f"results/eval_{arm}_r{r}_s{seed}{out_suffix}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in pr_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크 해제는 학습 직후로 옮겼다(R5). 여기서는 다음 라운드 기준만 갱신한다.
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None,
                 mmlu=True, max_new_tokens=256, out_suffix=""):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군.
       mmlu — ★R4 신설. arm0의 limit=10 MMLU는 이미 있다. 다시 돌리면 20분을 버리고
              results/mmlu_arm0_r0/ 에 타임스탬프 파일이 하나 더 쌓인다.
       max_new_tokens, out_suffix — ★R5 신설. 생성 상한과 충돌 없는 파일 접미사."""
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
    check(f"results/eval_arm0_r0_s{seed}{out_suffix}.json")
    if mmlu:
        run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")
    else:
        print("[skip] MMLU arm0 — 이미 측정됨(limit=10)")

print("정의 완료")

## 4. Jobs — change only JOB

Every non-baseline cell is one uninterrupted sequence:

train r1→r3 → harmful 7 lenses → IFEval 5 lenses → checkpoint fingerprint gate → cleanup

| JOB | seed | arms | purpose |
|---:|---:|---|---|
| 1 | 42 | arm0 + arm1 | baseline once, then primary agreeable cell |
| 2 | 42 | arm2 | seed-42 generic-FT control |
| 3 | 42 | arm3 | seed-42 principled control |
| 4–6 | 1337 | arm1 / arm2 / arm3 | separate conditional replication jobs |
| 7–9 | 2718 | arm1 / arm2 / arm3 | separate conditional replication jobs |

arm0 is evaluated once, only in JOB 1. Jobs 4–9 are not pooled into the
dual-primary analyzer; they are preserved as this benign-outcome cohort.

In [ ]:
import json, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
UNSLOTH_PIN = "2026.7.6"
NB_NAME = "nb_r13_benign"
OUT_SUFFIX = "_r13_benign_env76"
HARMFUL_LENSES = ['none', 'agreeable', 'principled', 'agreeable_para1', 'agreeable_para2', 'agreeable_weak', 'agreeable_strong']
BENIGN_LENSES = ['none', 'agreeable_weak', 'agreeable', 'agreeable_strong', 'principled']
MAX_NEW_TOKENS = 256
IFEVAL_LIMIT = 50
IFEVAL_BATCH_SIZE = 2
ADAPTER_ROOT = "/kaggle/working/adapters_r13_benign"
RUN_SMOKE = False
JOB = 1
JOB_MAP = {1: {'seed': 42, 'arms': ['arm0', 'arm1'], 'label': 'seed42-arm0-arm1'}, 2: {'seed': 42, 'arms': ['arm2'], 'label': 'seed42-arm2'}, 3: {'seed': 42, 'arms': ['arm3'], 'label': 'seed42-arm3'}, 4: {'seed': 1337, 'arms': ['arm1'], 'label': 'seed1337-arm1'}, 5: {'seed': 1337, 'arms': ['arm2'], 'label': 'seed1337-arm2'}, 6: {'seed': 1337, 'arms': ['arm3'], 'label': 'seed1337-arm3'}, 7: {'seed': 2718, 'arms': ['arm1'], 'label': 'seed2718-arm1'}, 8: {'seed': 2718, 'arms': ['arm2'], 'label': 'seed2718-arm2'}, 9: {'seed': 2718, 'arms': ['arm3'], 'label': 'seed2718-arm3'}}

cfg = json.load(open("configs/arms.json", encoding="utf-8"))
assert cfg["base_model"] == BASE, f"Dataset base model={cfg['base_model']} != {BASE}"
assert cfg["n_rounds"] == 3 and cfg["gen_batch_size"] == 16
assert OUT_SUFFIX == "_r13_benign_env76"
assert HARMFUL_LENSES == [
    "none", "agreeable", "principled", "agreeable_para1",
    "agreeable_para2", "agreeable_weak", "agreeable_strong",
]
assert BENIGN_LENSES == [
    "none", "agreeable_weak", "agreeable", "agreeable_strong", "principled",
]
assert len(set(JOB_MAP)) == 9 and JOB in JOB_MAP

from common import env_versions
RUNTIME_ENV = env_versions()
assert RUNTIME_ENV.get("unsloth") == UNSLOTH_PIN, (
    f"Unsloth 실행 환경 불일치: 기대={UNSLOTH_PIN} stamp={RUNTIME_ENV}")
print(f"[gate] env={RUNTIME_ENV} | job={JOB}/{JOB_MAP[JOB]}")
print(f"[gate] harmful={HARMFUL_LENSES} | benign={BENIGN_LENSES}")


## 5. Optional smoke path

Set RUN_SMOKE = True to run all five lens commands on only two IFEval items.
Smoke output is under results/_smoke/ and is never a production result or manifest.

In [ ]:
if RUN_SMOKE:
    run(
        f"python src/run_ifeval_lensed.py --model_path {BASE} "
        f"--arm arm0 --round 0 --seed 42 --dry_run"
    )
    print("[smoke] separate _smoke path completed; set RUN_SMOKE=False for a job")


## 6. Main job

The benign runner uses lm_eval --tasks ifeval --limit 50 --batch_size 2
--apply_chat_template --log_samples and takes its five system prompts from
configs/arms.json. --num_fewshot is intentionally absent: this is zero-shot.

In [ ]:
import json, os, shutil

job = JOB_MAP[JOB]
seed = job["seed"]
arms = job["arms"]
print(f"R13 benign JOB {JOB}: {job['label']} | {arms}")

def harmful_path(arm, rnd):
    rtag = "0" if arm == "arm0" else str(rnd)
    return f"results/eval_{arm}_r{rtag}_s{seed}{OUT_SUFFIX}.json"

def benign_manifest_path(arm, rnd):
    rtag = "0" if arm == "arm0" else str(rnd)
    return f"results/ifeval_manifest_{arm}_r{rtag}_s{seed}{OUT_SUFFIX}.json"

def run_benign(arm, rnd, model_path):
    run(
        f"python src/run_ifeval_lensed.py --model_path {model_path} "
        f"--arm {arm} --round {rnd} --seed {seed} "
        f"--harmful_result {harmful_path(arm, rnd)} "
        f"--adapter_root {ADAPTER_ROOT}"
    )

def snap(tag):
    zp = shutil.make_archive(
        f"/kaggle/working/r13_benign_results_{tag}",
        "zip",
        "/kaggle/working/experiment/results",
    )
    print(f"[snap] {zp} | {os.path.getsize(zp)/1e6:.2f} MB", flush=True)

if not RUN_SMOKE:
    for arm in arms:
        if arm == "arm0":
            print(f"\n===== arm0 x seed {seed} — harmful + benign once =====", flush=True)
            run_baseline(
                seed,
                lenses=",".join(HARMFUL_LENSES),
                mmlu=False,
                max_new_tokens=MAX_NEW_TOKENS,
                out_suffix=OUT_SUFFIX,
            )
            run_benign(arm, 0, BASE)
            snap(f"job{JOB}_arm0_s{seed}")
            continue

        print(f"\n===== {arm} x seed {seed} — train + paired evaluations =====", flush=True)
        ckpt = run_session(
            arm,
            seed,
            lenses=",".join(HARMFUL_LENSES),
            eval_rounds=(3,),
            probe=False,
            mmlu=False,
            max_new_tokens=MAX_NEW_TOKENS,
            out_suffix=OUT_SUFFIX,
            free_ckpt=False,
            save_adapters=ADAPTER_ROOT,
        )
        assert ckpt and os.path.exists(ckpt), (
            f"run_session did not return the round-3 checkpoint: {ckpt}"
        )
        try:
            run_benign(arm, 3, ckpt)
        finally:
            if os.path.exists(ckpt):
                shutil.rmtree(ckpt)
                print(f"[disk] deleted after harmful + benign evaluations: {ckpt}")
        snap(f"job{JOB}_{arm}_s{seed}")


## 7. Recovery archive

The results archive includes the new harmful-grid JSON, benign manifests, and
per-lens --log_samples files. The adapter archive is separate and is retained
for any later re-evaluation without retraining.

In [ ]:
import glob, os, shutil

tag = f"job{JOB}"
results_zip = shutil.make_archive(
    f"/kaggle/working/r13_benign_results_{tag}",
    "zip",
    "/kaggle/working/experiment/results",
)
print("results:", results_zip, "| size MB:", round(os.path.getsize(results_zip) / 1e6, 2))
if os.path.exists(ADAPTER_ROOT):
    adapters_zip = shutil.make_archive(
        f"/kaggle/working/r13_benign_adapters_{tag}",
        "zip",
        ADAPTER_ROOT,
    )
    print("adapters:", adapters_zip, "| size MB:", round(os.path.getsize(adapters_zip) / 1e6, 2))
else:
    print("[info] JOB {JOB} has no trained arm adapter (arm0 only)")
for path in sorted(glob.glob("results/ifeval_manifest_*.json")):
    print("manifest:", path)


## 8. Gate — retained model path and recorded fingerprint provenance

The gate checks the seven harmful lenses, five benign lenses, fixed n=50,
per-lens sample logs, and that the benign manifest copied the harmful fingerprint
while evaluating the same retained model path before cleanup. It does not claim an
independent benign fingerprint measurement. No dual-primary analysis is called.

In [ ]:
import json, os

from run_ifeval_lensed import assert_same_checkpoint

if RUN_SMOKE:
    print("[gate] smoke path only — production manifest gate skipped")
else:
    job = JOB_MAP[JOB]
    seed = job["seed"]
    for arm in job["arms"]:
        rnd = 0 if arm == "arm0" else 3
        hp = harmful_path(arm, rnd)
        mp = benign_manifest_path(arm, rnd)
        assert os.path.exists(hp), f"harmful result missing: {hp}"
        assert os.path.exists(mp), f"benign manifest missing: {mp}"
        harmful = json.load(open(hp, encoding="utf-8"))
        benign = json.load(open(mp, encoding="utf-8"))
        assert harmful["schema"] == "r2"
        assert harmful["n_prompts"] == 313
        assert harmful["lenses"] == HARMFUL_LENSES
        assert harmful["out_suffix"] == OUT_SUFFIX
        assert benign["schema"] == "r13_benign_ifeval_manifest_v1"
        assert benign["source_label"] == "r13_benign_ifeval_lens_conditioned"
        assert benign["lenses"] == BENIGN_LENSES
        assert benign["limit"] == IFEVAL_LIMIT == 50
        assert benign["n_prompts"] == 50
        assert benign["batch_size"] == IFEVAL_BATCH_SIZE == 2
        assert benign["num_fewshot"] == 0 and benign["zero_shot"]
        assert benign["log_samples"] and benign["load_in_4bit"]
        source = benign["checkpoint_fingerprint_source"]
        assert source["kind"] == "copied_from_harmful_result"
        assert source["path"] == benign["harmful_result"]
        assert source["field"] == "ckpt_fingerprint"
        assert not benign["checkpoint_fingerprint_independently_recomputed"]
        assert benign["same_model_path_before_cleanup"] is True
        evidence = benign["same_model_path_evidence"]
        assert evidence["harmful_result_model_path"] == harmful["model_path"]
        assert evidence["benign_runner_model_path"] == harmful["model_path"]
        assert len(benign["sample_logs"]) == len(BENIGN_LENSES)
        assert len(set(row["output_dir"] for row in benign["sample_logs"])) == 5
        assert len(set(row["sample_log"] for row in benign["sample_logs"])) == 5
        assert len(set(benign["prompt_hashes"].values())) == 1
        for row in benign["sample_logs"]:
            assert row["sample_count"] == 50
            assert os.path.exists(row["sample_log"])
        if arm != "arm0":
            assert benign["adapter_paths"], f"{arm}: saved adapter paths missing"
        fp = assert_same_checkpoint(harmful, benign)
        print(f"[gate] ✓ {arm} seed {seed} recorded harmful fp {fp[:16]}... "
              "on the same retained model path; not independently recomputed")
    print(f"[gate] R13 benign JOB {JOB} passed; not entering dual-primary analysis")
